# Phase 3 — Segmentation-Guided Prototype Classification (LG-PM, classification track)

**Question (RQ3/RQ4):** does a support-conditioned *predicted* nodule mask improve
benign/malignant classification over whole-image classification, and how much
performance is lost versus an oracle ground-truth lesion region?

**Design (three-way, RQ3 + error-propagation RQ4):**
1. **Whole-image ERM** — ImageNet-ResNet34 fine-tuned on TN5000 images (fair conventional baseline);
2. **Frozen-feature heads** on the *same* frozen Phase 1 encoder —
   - `R0 image` whole-image global pooling (no mask),
   - `R1p lesion` predicted-mask ROI pooling,
   - `R2p lesion+ring` predicted lesion **plus** peri-lesional context ring (LG-PM main),
   - `R1g GT box` oracle ROI from TN5000 expert boxes,
   - `R2g GT box+ring` oracle mirror of R2p;
3. **Class-prototype heads** (ProtoNet-style) with a nested per-class support sweep
   K ∈ {5, 10, 20, 50, 200, all}.

**Data:** TN5000 pathology-confirmed labels (Pascal-VOC XML, `"0"`=benign, `"1"`=malignant),
official 3500/500/1000 split. SHA-256 dedup inside TN5000 **and** against the TN3K support
pool. Val used only for early stopping, thresholds, and temperature; test untouched.

**Segments:** zero-shot — Phase 1 frozen encoder + Phase 2 fusion decoder, ONE fixed
K=20 TN3K-train support episode for every TN5000 image (deterministic).

**Prereqs (Kaggle):** internet ON, GPU T4; datasets `tn3k-data`, `tn5000`; Kaggle dataset(s)
containing `best_resnet34_seed42.pt` (Phase 1) and `best_fusion_decoder.pt` (Phase 2).

## 0. Configuration, protocol constants, and checkpoint discovery

In [ ]:
from pathlib import Path
import json, time, random, hashlib, warnings
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from IPython.display import display
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN = DEVICE.type == "cuda"

INPUT_ROOT  = Path("/kaggle/input/datasets")
AUTHOR_ROOT = INPUT_ROOT / "jonbon8397"
PATHS = {
    "tn3k":   AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "tn3k",
    "tn5000": AUTHOR_ROOT / "tn5000" / "TN5000_forReview",
}
OUTPUT_ROOT = Path("/kaggle/working/phase3_classification")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ---- protocol (fixed before any evaluation; consistent with Phases 1-2) ----
IMG_SIZE     = 384
MASK_THRESH  = 127
SEG_K        = 20                    # fixed support budget for TN5000 segmentation
PRED_THRESH  = 0.5
PROTO_SCALES = ["e2", "e3", "e4"]
SCORE_HW     = 48
RING_PX      = 15                    # context-ring dilation at 384^2
FEAT_SCALES  = ["e3", "e4"]          # pooled descriptor levels
BATCH        = 16
K_PROTO      = [5, 10, 20, 50, 200, "all"]   # per-class label support budgets
N_PROTO_DRAW = 25
RUN_ERM_FULL = True                  # ImageNet-ResNet34 whole-image fine-tune
ERM_EPOCHS, ERM_PATIENCE, ERM_LR, ERM_WD = 15, 4, 1e-4, 1e-4
LIN_EPOCHS, LIN_PATIENCE, LIN_LR, LIN_WD = 300, 25, 3e-3, 1e-2
N_BOOT       = 2000

ck_enc = list(Path("/kaggle/input").rglob("best_resnet34_seed42.pt")) +          list(Path("/kaggle/working").rglob("best_resnet34_seed42.pt"))
ck_dec = list(Path("/kaggle/input").rglob("best_fusion_decoder.pt")) +          list(Path("/kaggle/working").rglob("best_fusion_decoder.pt"))
assert ck_enc, ("Phase 1 checkpoint missing: upload best_resnet34_seed42.pt as a "
                "Kaggle dataset (from phase1 /kaggle/working), attach, re-run.")
assert ck_dec, ("Phase 2 checkpoint missing: upload best_fusion_decoder.pt as a "
                "Kaggle dataset (from /kaggle/working/phase2_prototype_seg/), attach, re-run.")
CKPT_ENC, CKPT_DEC = ck_enc[0], ck_dec[0]
print("device:", DEVICE, "\n  encoder:", CKPT_ENC, "\n  decoder:", CKPT_DEC)

## 1. Manifests — TN3K support pool, TN5000 official split (XML), dedup audit

TN5000 XML: Pascal-VOC; object `name` `"0"`=benign, `"1"`=malignant. Image label = 1 if
**any** object is malignant (documented rule); ROI box = union of the boxes of objects that
decide the label. Split from `ImageSets/Main/{train,val,val?,test}.txt` — NOT trainval.txt.
Dedup: SHA-256 inside TN5000 (priority train>val>test) **and** against TN3K train supports.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

# ---------- TN3K (support donors only; identical rules to Phases 1-2) ----------
def pair_images_masks(image_dir, mask_dir, dataset, split):
    imgs = {p.stem: p for p in image_files(image_dir)}
    msks = {p.stem: p for p in image_files(mask_dir)}
    common = sorted(set(imgs) & set(msks))
    return pd.DataFrame([{"dataset": dataset, "split": split, "id": k,
                          "image": str(imgs[k]), "mask": str(msks[k])} for k in common])

tn3k = pd.concat([
    pair_images_masks(PATHS["tn3k"]/"trainval-image", PATHS["tn3k"]/"trainval-mask", "TN3K", "trainval"),
    pair_images_masks(PATHS["tn3k"]/"test-image",     PATHS["tn3k"]/"test-mask",     "TN3K", "test"),
], ignore_index=True)
tn3k["sha256"] = tn3k["image"].apply(lambda p: sha256_file(p) if Path(p).exists() else None)
tn3k["_p"] = tn3k["split"].map({"trainval": 0, "test": 1})
tn3k = tn3k.sort_values(["_p", "id"], kind="stable").drop_duplicates("sha256", keep="first").drop(columns="_p")
rng = np.random.RandomState(SEED)
tv = tn3k[tn3k.split == "trainval"].copy()
perm = rng.permutation(len(tv))
val_ids = set(tv.iloc[perm[: int(round(0.15 * len(tv)))]].id)      # same val as Phases 1-2
tv["split"] = np.where(tv.id.isin(val_ids), "val", "train")
tn3k_train = tv[tv.split == "train"].reset_index(drop=True)
print("TN3K train support pool:", len(tn3k_train))

# ---------- TN5000 (classification; pathology labels + expert boxes) ----------
def parse_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    boxes, names = [], []
    for ob in root.findall("object"):
        nm = (ob.findtext("name") or "").strip().lower()
        bb = ob.find("bndbox")
        if bb is None:
            continue
        try:
            box = [float(bb.findtext(t)) for t in ("xmin", "ymin", "xmax", "ymax")]
        except (TypeError, ValueError):
            continue
        boxes.append(box); names.append(nm)
    return boxes, names

BENIGN, MALIGNANT = {"0", "benign"}, {"1", "malignant"}
def xml_to_row(xml_path):
    boxes, names = parse_xml(xml_path)
    if not boxes:
        return None
    mal = [i for i, n in enumerate(names) if n in MALIGNANT and n not in BENIGN]
    label = 1 if mal else 0
    use = [boxes[i] for i in (mal if mal else range(len(boxes)))]   # union of deciding boxes
    x0 = min(b[0] for b in use); y0 = min(b[1] for b in use)
    x1 = max(b[2] for b in use); y1 = max(b[3] for b in use)
    return {"label": label, "n_obj": len(boxes), "box": [x0, y0, x1, y1]}

t5_img = {p.stem: p for p in image_files(PATHS["tn5000"]/"JPEGImages")}
xml_dir = PATHS["tn5000"]/"Annotations"
xmls = {p.stem: p for p in xml_dir.glob("*.xml")} if xml_dir.exists() else {}
common = sorted(set(t5_img) & set(xmls))
assert common, "TN5000: no JPEG/XML name overlap - check mount paths."
rows, skipped = [], 0
for k in common:
    r = xml_to_row(xmls[k])
    if r is None:
        skipped += 1; continue
    rows.append({"dataset": "TN5000", "id": k, "image": str(t5_img[k]), **r})
tn5000 = pd.DataFrame(rows)
print(f"TN5000: {len(tn5000)} usable (skipped {skipped} XML with no valid box) | "
      f"multi-object imgs {(tn5000.n_obj > 1).sum()} | prevalence malignant "
      f"{tn5000.label.mean():.3f} (n={int(tn5000.label.sum())})")

# ---------- official split 3500/500/1000 (specific files, NOT trainval.txt) ----------
main_dir = PATHS["tn5000"]/"ImageSets"/"Main"
def read_split_file(stem):
    for cand in (main_dir/f"{stem}.txt", main_dir/f"{stem.upper()}.txt"):
        if cand.exists():
            ids = []
            for ln in cand.read_text().splitlines():
                ln = ln.strip()
                if not ln:
                    continue
                ids.append(Path(ln.split()[0]).stem)
            return ids, str(cand)
    return None, None
splits, used_files = {}, {}
for stem in ["train", "val", "test"]:
    ids, src = read_split_file(stem)
    if ids is not None:
        splits[stem] = ids; used_files[stem] = src
tn5000["split"] = None
if len(splits) == 3:
    for stem, ids in splits.items():
        tn5000.loc[tn5000.id.isin(ids), "split"] = stem
    print("official split files:", used_files)
else:
    print("WARNING: official split files not found - deterministic fallback 3500/500/1000")
    perm = np.random.RandomState(SEED).permutation(len(tn5000))
    idx = tn5000.index.to_numpy()[perm]
    tn5000.loc[idx[:3500], "split"] = "train"
    tn5000.loc[idx[3500:4000], "split"] = "val"
    tn5000.loc[idx[4000:], "split"] = "test"
_uncov = int(tn5000.split.isna().sum())
if _uncov:
    print(f"WARNING: {_uncov} TN5000 rows not covered by the split files - dropping them "
          f"(unlabelled for our purposes); coverage issue to report in the audit log")
    tn5000 = tn5000[tn5000.split.notna()].copy()
tn5000 = tn5000.reset_index(drop=True)

# ---------- dedup: inside TN5000 (train>val>test) and vs TN3K train ----------
tn5000["sha256"] = tn5000["image"].apply(sha256_file)
tn5000["_p"] = tn5000["split"].map({"train": 0, "val": 1, "test": 2})
before = len(tn5000)
tn5000 = tn5000.sort_values(["_p", "id"], kind="stable").drop_duplicates("sha256", keep="first")
tn5000 = tn5000.drop(columns="_p").reset_index(drop=True)
print(f"TN5000 dedup: {before} -> {len(tn5000)} (dropped {before - len(tn5000)} exact-duplicate images)")
ov = set(tn5000.sha256) & set(tn3k_train.sha256)
if ov:
    tn5000 = tn5000[~tn5000.sha256.isin(ov)].reset_index(drop=True)
print(f"cross-dataset overlap TN5000 vs TN3K-train: {len(ov)} (removed from TN5000)")

SPLITS = {s: tn5000[tn5000.split == s].reset_index(drop=True) for s in ["train", "val", "test"]}
for s, d in SPLITS.items():
    print(f"  {s:5s} n={len(d):5d}  malignant={int(d.label.sum()):4d} "
          f"({d.label.mean():.3f})")

## 2. Loading and preprocessing — 384 px letterbox (identical to Phases 1-2), box transform, context ring

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def read_gray(p):
    return np.asarray(Image.open(p).convert("L"))

def letterbox_image(img, size=IMG_SIZE):
    """img (H,W) raw 0-255 -> (ci float32 0-1 in size^2, geom dict for box mapping)."""
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    ri = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    t, l = (size - nh) // 2, (size - nw) // 2
    ci = np.zeros((size, size), np.float32); ci[t:t + nh, l:l + nw] = ri / 255.0
    return ci, {"s": s, "t": t, "l": l, "nh": nh, "nw": nw}

def box_to_384(box, geom):
    s, t, l = geom["s"], geom["t"], geom["l"]
    x0, y0, x1, y1 = box
    xa, ya = x0 * s + l, y0 * s + t
    xb, yb = x1 * s + l, y1 * s + t
    return [xa, ya, xb, yb]

def box_mask(box384, size=IMG_SIZE):
    m = np.zeros((size, size), np.uint8)
    x0, y0 = max(0, int(np.floor(box384[0]))), max(0, int(np.floor(box384[1])))
    x1, y1 = min(size, int(np.ceil(box384[2]))),  min(size, int(np.ceil(box384[3])))
    if x1 > x0 and y1 > y0:
        m[y0:y1, x0:x1] = 1
    return m

def make_ring(mask, r=RING_PX):
    """Peri-lesional context = dilated(mask, r) minus mask; empty -> image complement."""
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    dil = cv2.dilate(mask.astype(np.uint8), k)
    ring = np.clip(dil.astype(np.int16) - mask.astype(np.int16), 0, 1).astype(np.uint8)
    if ring.sum() == 0:
        ring = (1 - mask.astype(np.uint8))
    return ring

def to_tensor(img):
    x = torch.from_numpy(np.stack([img, img, img])).float()
    return (x - MEAN) / STD

def tn3k_support_rows(kmax=SEG_K, seed=SEED + 10):
    r = np.random.RandomState(seed)
    return tn3k_train.iloc[r.permutation(len(tn3k_train))[:kmax]].reset_index(drop=True)

def load_tn3k_item(row):
    img = read_gray(row["image"]).astype(np.float32)
    h, w = img.shape[:2]
    s = IMG_SIZE / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    ri = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    rm = cv2.resize((read_gray(row["mask"]) >= MASK_THRESH).astype(np.uint8), (nw, nh),
                    interpolation=cv2.INTER_NEAREST)
    t, l = (IMG_SIZE - nh) // 2, (IMG_SIZE - nw) // 2
    ci = np.zeros((IMG_SIZE, IMG_SIZE), np.float32); ci[t:t + nh, l:l + nw] = ri / 255.0
    cm = np.zeros((IMG_SIZE, IMG_SIZE), np.uint8);   cm[t:t + nh, l:l + nw] = (rm > 0).astype(np.uint8)
    return ci, cm

print("preprocessing ready | letterbox", IMG_SIZE, "| ring dilation +/-", RING_PX, "px")

## 3. Frozen LG-PM components — Phase 1 encoder (ResNet34-UNet, FROZEN) + Phase 2 fusion decoder
Architecture cells are exact copies of Phase 2 (state-dict compatible).

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = conv_block(cin + cskip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class Phase1UNet(nn.Module):
    """Identical architecture to the Phase 1 checkpoint (state-dict compatible)."""
    def __init__(self):
        super().__init__()
        import torchvision.models as tvm
        bb = tvm.resnet34(weights=None)
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu)
        self.pool = bb.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.up4, self.up3 = Up(512, 256, 256), Up(256, 128, 128)
        self.up2, self.up1 = Up(128, 64, 64), Up(64, 64, 64)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            conv_block(64, 32), nn.Conv2d(32, 1, 1))
    def features(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(self.pool(e0)); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        return {"e1": e1, "e2": e2, "e3": e3, "e4": e4}

backbone = Phase1UNet().to(DEVICE)
_ck = torch.load(CKPT_ENC, map_location=DEVICE)
backbone.load_state_dict(_ck["model"], strict=True)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad_(False)
print(f"loaded Phase 1 weights (which={_ck.get('which')}, val Dice={_ck.get('val_dice'):.4f}) -> encoder FROZEN")

@torch.no_grad()
def encode(x):
    with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        f = backbone.features(x)
    return {k: v.float() for k, v in f.items()}

_x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
_f = encode(_x)
print({k: tuple(v.shape) for k, v in _f.items()})

In [ ]:
def masked_proto(feats, masks):
    h, w = feats.shape[-2:]
    m = F.interpolate(masks, size=(h, w), mode="nearest")
    fg_den = m.sum(dim=(0, 2, 3)) + 1e-6
    bg_den = (1 - m).sum(dim=(0, 2, 3)) + 1e-6
    pf = (feats * m).sum(dim=(0, 2, 3)) / fg_den
    pb = (feats * (1 - m)).sum(dim=(0, 2, 3)) / bg_den
    return pf, pb

def neg_sq_dist(feats, proto):
    return -(feats - proto.view(1, -1, 1, 1)).pow(2).sum(1, keepdim=True)

def score_maps(que_feats, pf_pb_per_scale):
    outs = []
    for name in PROTO_SCALES:
        qf = que_feats[name]
        pf, pb = pf_pb_per_scale[name]
        outs.append(F.interpolate(neg_sq_dist(qf, pf), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
        outs.append(F.interpolate(neg_sq_dist(qf, pb), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
    return torch.cat(outs, dim=1)

class FusionDecoder(nn.Module):
    def __init__(self, cin=2 * len(PROTO_SCALES)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1))
    def forward(self, s):
        return self.net(s)

def predict_logits_384(que_feats, pf_pb_per_scale, decoder):
    return F.interpolate(decoder(score_maps(que_feats, pf_pb_per_scale)),
                         size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

decoder = FusionDecoder().to(DEVICE)
_dk = torch.load(CKPT_DEC, map_location=DEVICE)
decoder.load_state_dict(_dk["decoder"], strict=True)
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)
print(f"loaded Phase 2 fusion decoder (val Dice={_dk.get('val_dice'):.4f} @ ep {_dk.get('epoch')})")

# ---- ONE fixed K=20 support episode for all TN5000 images (deterministic) ----
SUP_ROWS = tn3k_support_rows(SEG_K)
_sup = [load_tn3k_item(r) for _, r in SUP_ROWS.iterrows()]
sup_x = torch.stack([to_tensor(i) for i, _ in _sup]).to(DEVICE)
sup_m = torch.stack([torch.from_numpy(m[None]) for _, m in _sup]).float().to(DEVICE)
with torch.no_grad():
    SF = encode(sup_x)
PF_PB = {name: masked_proto(SF[name], sup_m) for name in PROTO_SCALES}
print("fixed support episode: ids", list(SUP_ROWS.id)[:5], "... | fg pixel means:",
      {k: round(float(v[0].norm() / (v[0].abs() > 0).sum().clamp(min=1)), 4) for k, v in PF_PB.items()})

## 4. TN5000 zero-shot segmentation + ROI feature bank (single GPU pass)

For each image: predict mask (decoder, fixed supports), then build descriptors from the
**same** frozen encoder features (`e3`,`e4` masked global average pooling, L2-preserved):

| code | representation | region pooled |
|---|---|---|
| R0 image | whole image | everything (no mask) |
| R1p lesion | predicted mask | lesion only |
| R2p lesion+ring | predicted mask + ring | lesion + peri-lesional context (**main**) |
| R1g GT box | expert box (oracle) | lesion only |
| R2g GT box+ring | expert box + its ring (oracle) | mirror of R2p |

QC recorded per image: mask empty flag, area fraction, mean foreground probability.
Empty masks fall back to R0 pooling (flagged). Features + 96² masks cached to NPZ.

In [ ]:
def masked_gap(feat, msk01):
    h, w = feat.shape[-2:]
    m = F.interpolate(msk01, size=(h, w), mode="nearest")
    den = m.sum(dim=(2, 3)).clamp(min=1.0)            # (B,1) per-image masked area
    return (feat * m).sum(dim=(2, 3)) / den           # (B, C) per-image descriptor

def run_feature_bank(df, split):
    fpath = OUTPUT_ROOT / f"features_{split}.npz"
    if fpath.exists():
        z = np.load(fpath, allow_pickle=True)
        print(f"[cache] {split}: loaded {fpath.name}")
        return {k: z[k] for k in z.files}
    reps = ["R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtbox", "R2g_gtbox_ring"]
    store = {r: [] for r in reps}
    mask96, conf, area, empty, ids, labels = [], [], [], [], [], []
    t0 = time.time()
    for i0 in range(0, len(df), BATCH):
        chunk = df.iloc[i0:i0 + BATCH]
        imgs, gbox_m, geom = [], [], []
        for _, r in chunk.iterrows():
            ci, g = letterbox_image(read_gray(r["image"]).astype(np.float32))
            imgs.append(ci); geom.append(g)
            gbox_m.append(box_mask(box_to_384(r["box"], g)))
        x = torch.stack([to_tensor(i) for i in imgs]).to(DEVICE)
        with torch.no_grad():
            feats = encode(x)
            logits = predict_logits_384(feats, PF_PB, decoder)
            prob = torch.sigmoid(logits)
        pm = (prob >= PRED_THRESH).float()                                # (B,1,384,384)
        pm_raw = pm.clone()                                               # QC recorded pre-fallback
        for bi in range(len(chunk)):
            if pm[bi].sum() < 4:                                          # empty mask fallback
                pm[bi] = 1.0
                empty.append(1)
            else:
                empty.append(0)
        gtm = torch.from_numpy(np.stack(gbox_m)[:, None]).float().to(DEVICE)
        ring_p = torch.stack([torch.from_numpy(make_ring(m[0].cpu().numpy().astype(np.uint8))[None])
                              for m in pm]).float().to(DEVICE)
        ring_g = torch.stack([torch.from_numpy(make_ring(gbox_m[bi])[None])
                              for bi in range(len(chunk))]).float().to(DEVICE)
        masks = {"image": "image", "pred": pm, "gtbox": gtm, "ring_p": ring_p, "ring_g": ring_g}
        combo = {"R0_image": "image",
                 "R1p_lesion": "pred", "R2p_lesion_ring": ("pred", "ring_p"),
                 "R1g_gtbox": "gtbox", "R2g_gtbox_ring": ("gtbox", "ring_g")}
        desc = {}
        for rep, key in combo.items():
            if key == "image":
                desc[rep] = torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1).float().cpu()
            elif isinstance(key, tuple):
                a = torch.cat([masked_gap(feats[nm], masks[key[0]]) for nm in FEAT_SCALES], 1)
                b = torch.cat([masked_gap(feats[nm], masks[key[1]]) for nm in FEAT_SCALES], 1)
                desc[rep] = torch.cat([a, b], 1).float().cpu()
            else:
                desc[rep] = torch.cat([masked_gap(feats[nm], masks[key]) for nm in FEAT_SCALES], 1).float().cpu()
        for r_ in reps:
            store[r_].append(desc[r_].numpy())
        for bi in range(len(chunk)):
            m96 = cv2.resize(pm_raw[bi, 0].cpu().numpy().astype(np.uint8), (96, 96),
                             interpolation=cv2.INTER_NEAREST)
            mask96.append(m96)
            conf.append(float(prob[bi].mean()))
            area.append(float(pm_raw[bi].mean()))   # raw predicted mask area (pre-fallback)
        ids.extend(chunk.id.tolist()); labels.extend(chunk.label.astype(int).tolist())
        done = min(i0 + BATCH, len(df))
        if done % (BATCH * 25) < BATCH or done == len(df):
            print(f"  {split}: {done}/{len(df)} images ({(time.time()-t0)/60:.1f} min)")
    out = {"ids": np.array(ids), "labels": np.array(labels, np.int64),
           "mask96": np.stack(mask96), "conf": np.array(conf, np.float32),
           "area": np.array(area, np.float32), "empty": np.array(empty, np.int64)}
    for r_ in reps:
        out[r_] = np.concatenate(store[r_]).astype(np.float32)
    np.savez_compressed(fpath, **out)
    print(f"[done] {split}: n={len(ids)} | empty-mask fallback {sum(empty)}/{len(ids)} "
          f"({100*sum(empty)/max(1,len(ids)):.1f}%) | mean area {np.mean(area):.3f} | "
          f"mean conf {np.mean(conf):.3f} | {(time.time()-t0)/60:.1f} min")
    return out

BANK = {s: run_feature_bank(SPLITS[s], s) for s in ["train", "val", "test"]}

## 5. Baseline — ImageNet-ResNet34 whole-image ERM (the fair conventional classifier)
Fine-tuned on TN5000 train (BCE + pos_weight), early stop on val AUROC, seed 42.
This is the row a reviewer compares R2p against — it must be strong, not a strawman.

In [ ]:
class TN5KImageDS(Dataset):
    def __init__(self, df, train=False):
        self.df, self.train = df, train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ci, _ = letterbox_image(read_gray(r["image"]).astype(np.float32))
        if self.train:
            if np.random.rand() < 0.5:
                ci = np.ascontiguousarray(ci[:, ::-1])
            if np.random.rand() < 0.5:
                M = cv2.getRotationMatrix2D((IMG_SIZE/2, IMG_SIZE/2), np.random.uniform(-7, 7), 1.0)
                ci = cv2.warpAffine(ci, M, (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_LINEAR)
            if np.random.rand() < 0.5:
                ci = np.clip(ci * np.random.uniform(0.8, 1.2), 0, 1)
        return to_tensor(ci), np.float32(r["label"])

@torch.no_grad()
def predict_probs(model, df, bs=32):
    model.eval()
    ld = DataLoader(TN5KImageDS(df, False), batch_size=bs, shuffle=False,
                    num_workers=2, pin_memory=PIN)
    ps = []
    for x, _ in ld:
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            ps.append(torch.sigmoid(model(x)).squeeze(1).float().cpu())   # (B,) flat
    return torch.cat(ps).numpy()

ERM_PATH = OUTPUT_ROOT / "erm_resnet34_probs.json"
if RUN_ERM_FULL and not ERM_PATH.exists():
    import torchvision.models as tvm
    erm = tvm.resnet34(weights="DEFAULT")
    erm.fc = nn.Linear(erm.fc.in_features, 1)
    erm = erm.to(DEVICE)
    ytr = SPLITS["train"].label.to_numpy(np.float32)
    pos_w = torch.tensor([(len(ytr) - ytr.sum()) / max(ytr.sum(), 1)], device=DEVICE)
    opt = torch.optim.AdamW(erm.parameters(), lr=ERM_LR, weight_decay=ERM_WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ERM_EPOCHS)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    ld_tr = DataLoader(TN5KImageDS(SPLITS["train"], True), batch_size=32, shuffle=True,
                       num_workers=2, pin_memory=PIN, drop_last=True)
    best, wait, best_state = -1, 0, None
    t0 = time.time()
    for ep in range(1, ERM_EPOCHS + 1):
        erm.train(); tot, cnt = 0.0, 0
        for x, y in ld_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                loss = bce(erm(x).squeeze(1), y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += float(loss) * len(y); cnt += len(y)
        sched.step()
        pv = predict_probs(erm, SPLITS["val"])
        va = roc_auc_score(SPLITS["val"].label.to_numpy(), pv)
        star = ""
        if va > best + 1e-4:
            best, wait = va, 0
            best_state = {k: v.detach().cpu().clone() for k, v in erm.state_dict().items()}
            star = "  *"
        else:
            wait += 1
        print(f"erm ep {ep:02d} | train bce {tot/cnt:.4f} | val AUROC {va:.4f}{star} "
              f"({(time.time()-t0)/60:.1f} min)")
        if wait >= ERM_PATIENCE:
            print("erm early stop", ep); break
    erm.load_state_dict(best_state)
    probs = {s: predict_probs(erm, SPLITS[s]).tolist() for s in ["train", "val", "test"]}
    ERM_PATH.write_text(json.dumps({"val_auroc": best, "probs": probs}, indent=1))
    torch.save({"model": {k: v for k, v in best_state.items()},
                "val_auroc": best}, OUTPUT_ROOT / "erm_resnet34_tn5000.pt")
    print("ERM saved. best val AUROC", round(best, 4))
elif ERM_PATH.exists():
    print("[cache] ERM probs loaded:", ERM_PATH.name)
else:
    print("RUN_ERM_FULL=False - whole-image ERM row skipped")
ERM = json.loads(ERM_PATH.read_text()) if ERM_PATH.exists() else None

## 6. Linear heads on ROI descriptors (frozen features)

Each representation: standardise with **train** mean/std → logistic linear head
(AdamW, BCE + pos_weight), early stop on **val** AUROC. Reports which ROI matters,
with identical training for every representation — the only thing that changes is
*what is pooled*.

In [ ]:
def feat(rep, split):
    z = BANK[split][rep].astype(np.float32)
    return torch.from_numpy(z)

def labels(split):
    return BANK[split]["labels"].astype(np.float32)

LIN_PATH = OUTPUT_ROOT / "linear_heads_results.json"
def train_linear(rep):
    xtr, xva, xte = feat(rep, "train"), feat(rep, "val"), feat(rep, "test")
    ytr_t = torch.from_numpy(labels("train"))
    mu, sd = xtr.mean(0, keepdim=True), xtr.std(0, keepdim=True).clamp(min=1e-6)
    xtr, xva, xte = (xtr-mu)/sd, (xva-mu)/sd, (xte-mu)/sd
    xtr, xva, xte = xtr.to(DEVICE), xva.to(DEVICE), xte.to(DEVICE)
    ytr = ytr_t.to(DEVICE)
    lin = nn.Linear(xtr.shape[1], 1).to(DEVICE)
    pos_w = torch.tensor([(len(ytr_t)-ytr_t.sum())/max(ytr_t.sum(),1)], device=DEVICE)
    opt = torch.optim.AdamW(lin.parameters(), lr=LIN_LR, weight_decay=LIN_WD)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    best, wait, bw = -1, 0, None
    n = len(xtr)
    for ep in range(1, LIN_EPOCHS + 1):
        perm = torch.randperm(n, device=DEVICE)
        lin.train()
        for i0 in range(0, n, 512):
            idx = perm[i0:i0+512]
            loss = bce(lin(xtr[idx]).squeeze(1), ytr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        lin.eval()
        with torch.no_grad():
            pv = torch.sigmoid(lin(xva).squeeze(1)).cpu().numpy()
        va = roc_auc_score(labels("val"), pv)
        if va > best + 1e-4:
            best, wait = va, 0
            bw = {k: v.detach().cpu().clone() for k, v in lin.state_dict().items()}
        else:
            wait += 1
            if wait >= LIN_PATIENCE:
                break
    lin.load_state_dict(bw); lin.eval()
    with torch.no_grad():
        out = {s: torch.sigmoid(lin(x).squeeze(1)).cpu().numpy()
               for s, x in [("val", xva), ("test", xte)]}
    return {"val_auroc": float(best), "probs_val": out["val"].tolist(),
            "probs_test": out["test"].tolist(), "epochs": ep}

REPS = ["R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtbox", "R2g_gtbox_ring"]
if LIN_PATH.exists():
    LIN = json.loads(LIN_PATH.read_text())
    print("[cache] linear heads loaded")
else:
    LIN = {}
    for rep in REPS:
        t0 = time.time()
        LIN[rep] = train_linear(rep)
        print(f"{rep:18s} val AUROC {LIN[rep]['val_auroc']:.4f} "
              f"(ep {LIN[rep]['epochs']}, {time.time()-t0:.0f}s)")
    LIN_PATH.write_text(json.dumps(LIN, indent=1))

## 7. Class-prototype heads — lesion-grounded prototypes with nested label budgets

For each representation: L2-normalise descriptors, class-conditional means over
K per-class **train** cases (25 random draws; `all` = full train), score =
cos(mal) − cos(benign). This is the annotation-efficiency curve for the
*classification* label, parallel to the segmentation K-sweep of Phase 2.

In [ ]:
def l2n(x, eps=1e-8):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)

def proto_scores(rep, K, n_draw=N_PROTO_DRAW, seed=SEED):
    Xtr = l2n(BANK["train"][rep].astype(np.float64))
    ytr = BANK["train"]["labels"]
    Xte = l2n(BANK["test"][rep].astype(np.float64))
    rng = np.random.RandomState(seed)
    idx_b = np.where(ytr == 0)[0]; idx_m = np.where(ytr == 1)[0]
    aucs = []
    draws = [None] if K == "all" else range(n_draw)
    for d in draws:
        if K == "all":
            ib, im = idx_b, idx_m
        else:
            ib = rng.choice(idx_b, min(K, len(idx_b)), replace=False)
            im = rng.choice(idx_m, min(K, len(idx_m)), replace=False)
        pb, pm_ = Xtr[ib].mean(0), Xtr[im].mean(0)
        s = Xte @ pm_ - Xte @ pb
        aucs.append(roc_auc_score(BANK["test"]["labels"], s))
    return float(np.mean(aucs)), float(np.std(aucs))

rows = []
for rep in REPS:
    for K in K_PROTO:
        m_, s_ = proto_scores(rep, K)
        rows.append({"representation": rep, "K_per_class": K, "auroc_mean": round(m_, 4),
                     "auroc_sd": round(s_, 4), "draws": 1 if K == "all" else N_PROTO_DRAW})
PROTO_TAB = pd.DataFrame(rows)
PROTO_TAB.to_csv(OUTPUT_ROOT / "phase3_proto_support_sweep.csv", index=False)
piv = PROTO_TAB.pivot_table(index="representation", columns="K_per_class",
                            values="auroc_mean")[K_PROTO]
print("Prototype-head test AUROC (mean over 25 draws) by per-class support K:")
display(piv.round(4))

## 8. Main TN5000 test table — metrics, calibration, bootstrap CIs

Convention (fixed after phase3b audit): thresholds and temperature from **val only**
(spec≥0.90 threshold via val ROC). Threshold-based metrics (sens@spec≥0.90) use the
**raw-probability family on both sides** (raw-raw); Brier/ECE use the **temperature-
calibrated** probabilities. AUROC/AUPRC are rank metrics and unaffected. 95% CIs =
case-level bootstrap over test cases (2000).

In [ ]:
def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def fit_temperature(y, p):
    z, yv = logit(np.asarray(p)), np.asarray(y, np.float64)
    grid = np.exp(np.linspace(np.log(0.05), np.log(20.0), 200))
    def nll(T):
        q = 1 / (1 + np.exp(-z / T))
        return -np.mean(yv * np.log(q + 1e-12) + (1 - yv) * np.log(1 - q + 1e-12))
    return float(grid[int(np.argmin([nll(T) for T in grid]))])

def ece(y, p, bins=15):
    y = np.asarray(y, np.float64).ravel(); p = np.asarray(p, np.float64).ravel()
    edges = np.linspace(0, 1, bins + 1); out = 0.0
    for i in range(bins):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < bins - 1 else p <= edges[i + 1])
        if m.sum():
            out += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(out)

def spec90_threshold(y_val, p_val):
    fpr, tpr, thr = roc_curve(y_val, p_val)
    spec = 1 - fpr
    m = spec >= 0.90
    if m.any():
        i = np.where(m)[0][np.argmax(tpr[m])]
    else:
        i = int(np.argmax(spec))
    return float(thr[i]), float(tpr[i]), float(spec[i])

def brier(y, p):
    return float(np.mean((np.asarray(p).ravel() - np.asarray(y).ravel()) ** 2))

def metric_set(y, p, thr):
    y = np.asarray(y).ravel(); p = np.asarray(p).ravel()
    return {"auroc": roc_auc_score(y, p), "auprc": average_precision_score(y, p),
            "sens@spec90": float((p[y == 1] >= thr).mean()), "brier": brier(y, p),
            "ece": ece(y, p)}

def boot_ci(y, p, thr, n=N_BOOT, seed=SEED):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel()
    rng = np.random.RandomState(seed); N = len(y); acc = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2:
            continue
        acc.append([roc_auc_score(y[idx], p[idx]), average_precision_score(y[idx], p[idx]),
                    (p[idx][y[idx] == 1] >= thr).mean()])
    lo, hi = np.percentile(acc, 2.5, 0), np.percentile(acc, 97.5, 0)
    return lo, hi

MODEL_ROWS = {"ERM whole-image (ResNet34 ft)": ("erm", None),
              "R0 image | linear":  ("lin", "R0_image"),
              "R1p lesion | linear": ("lin", "R1p_lesion"),
              "R2p lesion+ring | linear (LG-PM)": ("lin", "R2p_lesion_ring"),
              "R1g GT box | linear (oracle)": ("lin", "R1g_gtbox"),
              "R2g GT box+ring | linear (oracle)": ("lin", "R2g_gtbox_ring")}

y_test = BANK["test"]["labels"]; y_val = BANK["val"]["labels"]
main_rows, TEMP = [], {}
for name, (kind, rep) in MODEL_ROWS.items():
    if kind == "erm":
        if ERM is None: continue
        pv, pt = np.array(ERM["probs"]["val"]).ravel(), np.array(ERM["probs"]["test"]).ravel()
    else:
        pv, pt = np.array(LIN[rep]["probs_val"]).ravel(), np.array(LIN[rep]["probs_test"]).ravel()
    T = fit_temperature(y_val, pv); TEMP[name] = T
    ptc = 1 / (1 + np.exp(-logit(pt) / T))
    thr, sens_v, spec_v = spec90_threshold(y_val, pv)          # operating point on RAW val probs
    ms = metric_set(y_test, pt, thr)                           # RAW-RAW: threshold + evaluation same family
    ms["brier"] = brier(y_test, ptc)                           # calibration metrics on calibrated probs
    ms["ece"] = ece(y_test, ptc)
    lo, hi = boot_ci(y_test, pt, thr)
    main_rows.append({"model": name, "auroc": round(ms["auroc"], 4),
                      "auroc_CI": f"({lo[0]:.4f}-{hi[0]:.4f})",
                      "auprc": round(ms["auprc"], 4), "sens@spec90(val)": round(ms["sens@spec90"], 4),
                      "brier": round(ms["brier"], 4), "ece_post": round(ms["ece"], 4),
                      "ece_pre": round(ece(y_test, pt), 4), "T_val": round(T, 3)})
MAIN = pd.DataFrame(main_rows)
MAIN.to_csv(OUTPUT_ROOT / "phase3_main_table.csv", index=False)
print("TN5000 test (n=%d, malignant=%d) - thresholds & temperature fitted on val:" %
      (len(y_test), int(y_test.sum())))
display(MAIN)

## 9. Error propagation (RQ4) — oracle ROI vs predicted ROI

Gap diagnostics: `R2g(GT box+ring) − R2p(pred lesion+ring)` = cost of segmentation error
in the classification head. Stratified by segmentation **confidence quartiles** (mean fg
probability) and mask **area quartiles**; empty-mask fallback rate included. Spearman
correlation between confidence and per-case correctness.

In [ ]:
from scipy.stats import spearmanr
p_pred = 1/(1+np.exp(-logit(np.array(LIN["R2p_lesion_ring"]["probs_test"]).ravel())/TEMP["R2p lesion+ring | linear (LG-PM)"]))
p_gt   = 1/(1+np.exp(-logit(np.array(LIN["R2g_gtbox_ring"]["probs_test"]).ravel())/TEMP["R2g GT box+ring | linear (oracle)"]))
conf, area, empty = BANK["test"]["conf"], BANK["test"]["area"], BANK["test"]["empty"]
correct_p = ((p_pred >= 0.5).astype(int) == y_test).astype(float)
gap_tbl = pd.DataFrame({
    "conf_quartile": pd.qcut(conf, 4, labels=["Q1(low)", "Q2", "Q3", "Q4(high)"]),
    "y_label": y_test, "p_pred": p_pred, "p_gt": p_gt})
rows = []
for q, g in gap_tbl.groupby("conf_quartile", observed=True):
    if g.y_label.nunique() < 2:
        continue
    rows.append({"conf quartile": str(q), "n": len(g),
                 "AUROC pred-ROI": round(roc_auc_score(g.y_label, g.p_pred), 4),
                 "AUROC GT-ROI": round(roc_auc_score(g.y_label, g.p_gt), 4),
                 "gap": round(roc_auc_score(g.y_label, g.p_gt)
                              - roc_auc_score(g.y_label, g.p_pred), 4)})
PROP = pd.DataFrame(rows)
rho, pval = spearmanr(conf, correct_p)
print(f"test empty-mask fallback rate: {empty.mean()*100:.2f}% ({int(empty.sum())}/{len(empty)})")
print(f"Spearman(mask confidence, prediction correct): rho={rho:.3f} (p={pval:.2e})")
display(PROP)
PROP.to_csv(OUTPUT_ROOT / "phase3_error_propagation.csv", index=False)

## 10. Calibration — val-fitted temperature, reliability on test
ECE before/after temperature; reliability curves for whole-image ERM, R0, and R2p.

In [ ]:
def reliability(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    xs, ys, ns = [], [], []
    for i in range(bins):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < bins - 1 else p <= edges[i + 1])
        if m.sum():
            xs.append(float(np.mean(p[m]))); ys.append(float(np.mean(y[m]))); ns.append(int(m.sum()))
    return xs, ys, ns

fig, ax = plt.subplots(figsize=(5.2, 5))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
cal_specs = [("ERM whole-image", "erm", None),
             ("R0 image | linear", "lin", "R0_image"),
             ("R2p lesion+ring (LG-PM)", "lin", "R2p_lesion_ring")]
for name, kind, rep in cal_specs:
    if kind == "erm":
        if ERM is None: continue
        pv, pt = np.array(ERM["probs"]["val"]).ravel(), np.array(ERM["probs"]["test"]).ravel()
    else:
        pv, pt = np.array(LIN[rep]["probs_val"]).ravel(), np.array(LIN[rep]["probs_test"]).ravel()
    T = fit_temperature(y_val, pv)
    xs, ys_, ns = reliability(y_test, 1/(1+np.exp(-logit(pt)/T)))
    ax.plot(xs, ys_, "o-", label=f"{name} (ECE {ece(y_test, 1/(1+np.exp(-logit(pt)/T))):.3f})")
ax.set_xlabel("mean predicted probability (test)"); ax.set_ylabel("observed malignant fraction")
ax.set_title("TN5000 test reliability (temperature fitted on val)")
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "phase3_reliability.png", dpi=140)
plt.show()

cal_rows = []
for name, (kind, rep) in MODEL_ROWS.items():
    if kind == "erm":
        if ERM is None: continue
        pv, pt = np.array(ERM["probs"]["val"]).ravel(), np.array(ERM["probs"]["test"]).ravel()
    else:
        pv, pt = np.array(LIN[rep]["probs_val"]).ravel(), np.array(LIN[rep]["probs_test"]).ravel()
    T = TEMP[name]
    cal_rows.append({"model": name, "ECE pre": round(ece(y_test, pt), 4),
                     "ECE post": round(ece(y_test, 1/(1+np.exp(-logit(pt)/T))), 4), "T": round(T, 3)})
CAL = pd.DataFrame(cal_rows)
CAL.to_csv(OUTPUT_ROOT / "phase3_calibration.csv", index=False)
display(CAL)

## 11. Qualitative panel — image, expert box, predicted mask + ring, calibrated probability

In [ ]:
rng_q = np.random.RandomState(SEED + 21)
dfq = SPLITS["test"]
pos_idx = dfq.index[dfq.label == 1].to_numpy(); neg_idx = dfq.index[dfq.label == 0].to_numpy()
pick = list(rng_q.choice(pos_idx, 3, replace=False)) + list(rng_q.choice(neg_idx, 3, replace=False))
id2row = {b: i for i, b in enumerate(BANK["test"]["ids"])}
pmap = {i_: float(p_pred[id2row[i_]]) for i_ in BANK["test"]["ids"]}
fig, axes = plt.subplots(len(pick), 3, figsize=(11, 3.1 * len(pick)))
for r_, i_ in enumerate(pick):
    row = dfq.loc[i_]
    ci, g = letterbox_image(read_gray(row["image"]).astype(np.float32))
    box = box_to_384(row["box"], g)
    m = cv2.resize(BANK["test"]["mask96"][id2row[row.id]], (IMG_SIZE, IMG_SIZE),
                   interpolation=cv2.INTER_NEAREST)
    ring = make_ring(m.astype(np.uint8))
    ov = np.dstack([ci, ci, ci]).copy()
    ov[..., 0] = np.clip(ov[..., 0] + 0.5 * m, 0, 1)
    ov[..., 2] = np.clip(ov[..., 2] + 0.3 * ring, 0, 1)
    axes[r_, 0].imshow(ci, cmap="gray"); axes[r_, 0].set_title(f"{row.id} label={int(row.label)}")
    axes[r_, 1].imshow(ci, cmap="gray")
    from matplotlib.patches import Rectangle
    axes[r_, 1].add_patch(Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                    fill=False, ec="lime", lw=1.5))
    axes[r_, 1].set_title("expert box (oracle ROI)")
    axes[r_, 2].imshow(ov)
    axes[r_, 2].set_title(f"mask+ring, p(mal)={pmap[row.id]:.2f}")
    for a in axes[r_]:
        a.axis("off")
plt.suptitle("TN5000 test: lesion-grounded ROI (red=lesion, blue=ring) vs oracle box", y=1.0)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "phase3_qualitative.png", dpi=140, bbox_inches="tight")
plt.show()

## 12. Run summary and artifacts

In [ ]:
run = {"phase": "3 - segmentation-guided classification",
       "seed": SEED, "device": str(DEVICE),
       "checkpoints": {"encoder": str(CKPT_ENC), "decoder": str(CKPT_DEC)},
       "config": {"IMG_SIZE": IMG_SIZE, "SEG_K": SEG_K, "PRED_THRESH": PRED_THRESH,
                  "RING_PX": RING_PX, "FEAT_SCALES": FEAT_SCALES,
                  "support_episode_ids": list(SUP_ROWS.id),
                  "K_PROTO": [str(k) for k in K_PROTO], "N_BOOT": N_BOOT,
                  "pos_weight_train": float((len(SPLITS['train'])-SPLITS['train'].label.sum())
                                            / max(SPLITS['train'].label.sum(), 1))},
       "split_sizes": {s: len(SPLITS[s]) for s in SPLITS},
       "erm_val_auroc": (ERM or {}).get("val_auroc"),
       "main_table": MAIN.to_dict(orient="records")}
with open(OUTPUT_ROOT / "run_summary.json", "w") as f:
    json.dump(run, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Artifacts in", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.iterdir()):
    print(f"  {p.name:44s} {p.stat().st_size/1024:9.1f} KB")

## 13. Decision log and hand-off

**What the paper reads from this run**
- Row *R0 image | linear* vs *R2p lesion+ring | linear*: the lesion-grounding effect with
  identical features and identical training (the clean comparison).
- Row *ERM whole-image*: the fair conventional baseline (full fine-tune).
- Rows *R1g/R2g (oracle)*: the no-segmentation-error upper bound; `oracle − R2p` =
  segmentation-error cost (with §9 confidence stratification and empty-mask rate).
- §7: classification-label annotation-efficiency curve (K per class).
- §8/§10: AUROC/AUPRC/sens@spec90 with bootstrap CIs; ECE pre/post **val-fitted** temperature.

**Send back (one by one):**
1. §4 feature-bank log (empty-mask rates, mean area/conf, per-split times);
2. §5 ERM training log + best val AUROC;
3. §6 linear heads val AUROC table;
4. §7 prototype support-sweep pivot;
5. §8 main test table (+ §10 calibration table);
6. §9 error-propagation table + Spearman ρ and empty-mask rate;
7. §11 qualitative figure description; runtime of the ERM section.

**Design boundaries preserved:** test untouched until §8; thresholds/temperature from val;
dedup inside TN5000 and vs TN3K supports documented; one fixed support episode released
(`support_episode_ids` in run_summary.json) for reproducibility.